In [ ]:
!pip install uv

In [ ]:
!uv pip install --system \
  transformers==4.56.1 \
  trl==0.23.1 \
  peft==0.14.0 \
  accelerate==1.6.0 \
  datasets==3.2.0 \
  bitsandbytes \
  sentencepiece \
  protobuf \
  scipy \
  einops \
  ninja \
  packaging \
  wheel

In [ ]:
!uv pip install --system --no-cache \
  "unsloth==2026.3.7" \
  "unsloth_zoo>=2026.3.4"

In [ ]:
!uv pip install --system \
  numpy==2.0.2 \
  pandas==2.2.2 \
  pyarrow

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp

%cd /content/llama.cpp

!apt-get update -y && apt-get install -y cmake build-essential

!cmake -B build
!cmake --build build --config Release --target llama-quantize -j 4

In [ ]:
import os
import sys
import torch
import subprocess

from unsloth import FastLanguageModel

from google.colab import drive
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer
from peft import LoraConfig

In [ ]:
# ======================
# -- PRESETS
# ======================

drive.mount('/content/drive')

os.environ['WANDB_MODE'] = 'disabled'

# ======================
# -- PATHS
# ======================

MODEL_ID   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_ID = "g0ofycatz/OkinDataset"

BASE_DIR   = "/content/drive/MyDrive/okin"
OUTPUT_DIR = os.path.join(BASE_DIR, "okin_llm")
GGUF_DIR   = os.path.join(BASE_DIR, "gguf")
LLAMA_CPP  = "/content/llama.cpp"
QUANT_TYPE  = "Q4_K_M"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

# ======================
# MODEL
# ======================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing="unsloth",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

tokenizer.pad_token = tokenizer.eos_token

# ======================
# -- DATA
# ======================

dataset = load_dataset(DATASET_ID, split="train")

def formatting_func(example):
    messages = example["messages"]

    system = ""
    user = ""
    assistant = ""

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if role == "system":
            system = content
        elif role == "user":
            user = content
        elif role == "assistant":
            assistant = content

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": prompt}

dataset = dataset.map(formatting_func)

# ======================
# -- TRAIN
# ======================

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    packing=True,
    group_by_length=True,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=25,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=1,s
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    max_length=512,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=args,
)

FastLanguageModel.for_training(model)

trainer.train()

merged_dir = os.path.join(OUTPUT_DIR, "merged")

model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit",
)

print("Saved merged model:", merged_dir)

# ======================
# -- GGUF
# ======================

convert_script = os.path.join(
    LLAMA_CPP,
    "convert_hf_to_gguf.py",
)

gguf_fp16 = os.path.join(
    GGUF_DIR,
    "okin-qwen-f16.gguf",
)

subprocess.run(
    [
        sys.executable,
        convert_script,
        merged_dir,
        "--outfile",
        gguf_fp16,
        "--outtype",
        "f16",
    ],
    check=True,
)

print("FP16 GGUF saved:", gguf_fp16)

# ======================
# QUANTIZE
# ======================

quantized_path = os.path.join(
    GGUF_DIR,
    f"okin-qwen-{QUANT_TYPE.lower()}.gguf",
)

quantize_bin = os.path.join(
    LLAMA_CPP,
    "build",
    "bin",
    "llama-quantize",
)

subprocess.run(
    [
        quantize_bin,
        gguf_fp16,
        quantized_path,
        QUANT_TYPE,
    ],
    check=True,
)

print("Quantized GGUF saved:", quantized_path)